# Phase 2 — Build & Evaluate the RAG Pipeline
### Commercial Contract Review Assistant — CUAD Dataset

This notebook builds and evaluates a retrieval-augmented generation (RAG) pipeline
over the **Contract Understanding Atticus Dataset (CUAD)**.

**Legal disclaimer:** This is an educational/research project. Any answer generated by
this system is derived only from retrieved contract text and is **not legal advice**.

**Engineering ground rule:** every statistic, count, or evaluation result below comes
from code executed in this notebook. Nothing is hard-coded or invented. If you re-run
this notebook on a different copy of CUAD, the numbers (and the narrative text that
reports them, where built from f-strings) will update automatically.

### Validation status of this notebook

The **code below is production code**, written to run against the real CUAD dataset
at `data/CUAD/CUAD_v1/full_contract_pdf/`. It contains no dummy-data-specific logic.

Before delivery, this notebook was executed end-to-end in a sandboxed development
environment to catch bugs, using **synthetic placeholder contracts** (not real CUAD
data), because that sandbox has no internet access to Hugging Face Hub (to download
`all-MiniLM-L6-v2`) and no local Ollama service. Concretely:

| Component | Validated in sandbox? | Notes |
|---|---|---|
| PDF discovery, parsing, page/empty/OCR stats (2.1) | Yes, on dummy PDFs | Real logic, real execution |
| Chunking with deterministic IDs (2.2) | Yes, on dummy PDFs | Real logic, real execution |
| ChromaDB build/load, persistence (2.3) | Yes, structurally | A substitute embedder was used only because the real model couldn't be downloaded in that sandbox |
| Embedding with `all-MiniLM-L6-v2` (2.3) | **NOT EXECUTED IN SANDBOX — REQUIRES LOCAL ENVIRONMENT** | No Hugging Face Hub access in that sandbox; will download/run normally on your machine |
| Retrieval function & prompt construction (2.4) | Yes, structurally | Real logic; result *quality* depends on the real embedding model above |
| Ollama generation (2.4) | **NOT EXECUTED IN SANDBOX — REQUIRES LOCAL ENVIRONMENT** | No Ollama service running in that sandbox; the notebook correctly detected this and skipped generation gracefully rather than faking it |
| Evaluation / Hit@K / failure analysis (2.6) | Yes, structurally, on dummy data + dummy ground truth | Numbers from the dummy dry-run are **not representative** of real CUAD performance |
| Config export & persistence verification (2.7) | Yes | Real logic, real execution |

**This notebook has not yet been run against your real CUAD dataset.** Any numbers
you may see from a sandboxed dry-run were produced from synthetic test contracts made
up purely to exercise the code paths — they say nothing about real retrieval quality.
Run this notebook in your own environment (with internet access and, optionally, a
running Ollama instance) to get real, trustworthy statistics and evaluation results.

## 0. Environment Inspection

Before writing the pipeline, we check the Python version and confirm which required
packages are actually importable in this environment. We do **not** reinstall or
upgrade anything that already works.

In [1]:
import sys
import importlib
import platform

print(f"Python version: {platform.python_version()}")
print(f"Executable: {sys.executable}")
print("-" * 60)

REQUIRED_PACKAGES = [
    "pandas",
    "numpy",
    "chromadb",
    "sentence_transformers",
    "pypdf",
    "ollama",
    "dotenv",  # python-dotenv
]

availability = {}
for pkg in REQUIRED_PACKAGES:
    try:
        module = importlib.import_module(pkg)
        version = getattr(module, "__version__", "unknown")
        availability[pkg] = version
        print(f"[OK]      {pkg:<22} version={version}")
    except ImportError as e:
        availability[pkg] = None
        print(f"[MISSING] {pkg:<22} -> {e}")

missing = [p for p, v in availability.items() if v is None]
if missing:
    print("\nThe following packages are missing and must be installed before "
          f"continuing: {missing}")
else:
    print("\nAll required packages are available. No installation needed.")

Python version: 3.13.0
Executable: c:\Users\Lenovo\rag-assistant-project\.venv\Scripts\python.exe
------------------------------------------------------------
[OK]      pandas                 version=3.0.5
[OK]      numpy                  version=2.5.3
[OK]      chromadb               version=1.5.9
[OK]      sentence_transformers  version=6.0.1
[OK]      pypdf                  version=6.18.0
[OK]      ollama                 version=unknown
[OK]      dotenv                 version=unknown

All required packages are available. No installation needed.


## 1. Setup & Configuration

All paths are resolved dynamically relative to this notebook's location (via
`pathlib.Path`), so the project remains portable across machines (including the
Windows path this project actually lives at,
`C:\Users\Lenovo\rag-assistant-project`, or any other checkout location).

No values here are hard-coded assumptions about dataset size — they are just the
pipeline's tunable configuration. Dataset statistics are computed later, from disk.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os

# --- Resolve project root dynamically -------------------------------------
# This notebook lives at <PROJECT_ROOT>/notebooks/rag_pipeline.ipynb
try:
    NOTEBOOK_DIR = Path(globals()["__vsc_ipynb_file__"]).resolve().parent
except KeyError:
    NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    # Fallback: assume we're already at the project root (e.g. running from CWD)
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data"
CUAD_DIR = DATA_DIR / "CUAD" / "CUAD_v1"
PDF_DIR = CUAD_DIR / "full_contract_pdf"
TXT_DIR = CUAD_DIR / "full_contract_txt"
MASTER_CLAUSES_CSV = CUAD_DIR / "master_clauses.csv"
CUAD_JSON_PATH = CUAD_DIR / "CUAD_v1.json"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
VECTOR_STORE_CONFIG_PATH = VECTOR_STORE_DIR / "config.json"

COLLECTION_NAME = "cuad_contracts"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Chunking configuration (see Section 2.2 for the rationale behind these values)
CHUNK_SIZE_TOKENS = 1500       # approx. tokens per chunk
CHUNK_OVERLAP_TOKENS = 200     # approx. token overlap between adjacent chunks
CHARS_PER_TOKEN_ESTIMATE = 4   # rough heuristic for English legal text

TOP_K = 5

OLLAMA_MODEL = None  # resolved dynamically in Section 2.4 based on what's installed

print(f"PROJECT_ROOT           = {PROJECT_ROOT}")
print(f"DATA_DIR                = {DATA_DIR}")
print(f"PDF_DIR                 = {PDF_DIR}")
print(f"TXT_DIR                 = {TXT_DIR}")
print(f"MASTER_CLAUSES_CSV      = {MASTER_CLAUSES_CSV}")
print(f"CUAD_JSON_PATH          = {CUAD_JSON_PATH}")
print(f"VECTOR_STORE_DIR        = {VECTOR_STORE_DIR}")
print(f"COLLECTION_NAME         = {COLLECTION_NAME}")
print(f"EMBEDDING_MODEL_NAME    = {EMBEDDING_MODEL_NAME}")
print(f"CHUNK_SIZE_TOKENS       = {CHUNK_SIZE_TOKENS}")
print(f"CHUNK_OVERLAP_TOKENS    = {CHUNK_OVERLAP_TOKENS}")
print(f"TOP_K                   = {TOP_K}")

for p, label in [(PDF_DIR, "PDF_DIR"), (MASTER_CLAUSES_CSV, "MASTER_CLAUSES_CSV"),
                  (CUAD_JSON_PATH, "CUAD_JSON_PATH")]:
    exists = p.exists()
    print(f"\nExists? {label}: {exists} -> {p}")
    if not exists:
        print(f"  WARNING: expected path not found. Update PROJECT_ROOT/paths above "
              f"or confirm the dataset location before proceeding.")

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT           = C:\Users\Lenovo\rag-assistant-project
DATA_DIR                = C:\Users\Lenovo\rag-assistant-project\data
PDF_DIR                 = C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\full_contract_pdf
TXT_DIR                 = C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\full_contract_txt
MASTER_CLAUSES_CSV      = C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\master_clauses.csv
CUAD_JSON_PATH          = C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\CUAD_v1.json
VECTOR_STORE_DIR        = C:\Users\Lenovo\rag-assistant-project\data\vector_store
COLLECTION_NAME         = cuad_contracts
EMBEDDING_MODEL_NAME    = all-MiniLM-L6-v2
CHUNK_SIZE_TOKENS       = 1500
CHUNK_OVERLAP_TOKENS    = 200
TOP_K                   = 5

Exists? PDF_DIR: True -> C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\full_contract_pdf

Exists? MASTER_CLAUSES_CSV: True -> C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\master_clauses.csv

Exis

## 2.1 Load & Inspect

We recursively scan `full_contract_pdf/` for every PDF, extract text with `pypdf`,
and record per-document/per-page metadata. From this we compute real corpus
statistics — nothing below is assumed in advance.

Documents get **deterministic IDs** (`CUAD_000001`, `CUAD_000002`, ...) assigned by
sorting discovered file paths, so re-running this notebook on the same directory
always produces the same IDs. This matters for reproducible retrieval citations and
evaluation.

In [3]:
from pypdf import PdfReader
from pypdf.errors import PdfReadError

def infer_contract_type(pdf_path: Path, pdf_root: Path) -> str:
    '''Infer a coarse contract type from the folder structure under PDF_DIR.
    CUAD organizes contracts into category subfolders (e.g. Part_I/Marketing).
    We use the immediate parent folder name as a lightweight type label.
    '''
    try:
        rel_parts = pdf_path.relative_to(pdf_root).parts
    except ValueError:
        rel_parts = pdf_path.parts
    # Prefer the last folder before the filename, skipping generic "Part_*" wrappers
    candidates = [p for p in rel_parts[:-1] if not p.lower().startswith("part_")]
    if candidates:
        return candidates[-1]
    elif len(rel_parts) > 1:
        return rel_parts[-2]
    return "Unknown"


def is_ocr_candidate(pages_text, empty_page_count, total_pages) -> bool:
    '''Heuristic: a document is a likely OCR candidate if it has pages but a large
    fraction of them extracted no usable text (common with scanned/image-only PDFs).
    '''
    if total_pages == 0:
        return False
    return (empty_page_count / total_pages) >= 0.5


records = []
page_records = []
failures = []

if not PDF_DIR.exists():
    raise FileNotFoundError(
        f"PDF_DIR does not exist: {PDF_DIR}. Confirm the CUAD dataset location."
    )

pdf_paths = sorted(PDF_DIR.rglob("*.pdf"))
print(f"Found {len(pdf_paths)} PDF files under {PDF_DIR}")

for idx, pdf_path in enumerate(pdf_paths, start=1):
    document_id = f"CUAD_{idx:06d}"
    contract_type = infer_contract_type(pdf_path, PDF_DIR)

    doc_record = {
        "document_id": document_id,
        "filename": pdf_path.name,
        "source_path": str(pdf_path.relative_to(PROJECT_ROOT)),
        "contract_type": contract_type,
        "num_pages": 0,
        "num_empty_pages": 0,
        "total_chars": 0,
        "parse_status": "success",
        "parse_error": None,
        "is_ocr_candidate": False,
    }

    try:
        reader = PdfReader(str(pdf_path))
        num_pages = len(reader.pages)
        doc_record["num_pages"] = num_pages
        empty_pages = 0

        for page_num, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""
            except Exception as page_exc:
                text = ""
            stripped = text.strip()
            if not stripped:
                empty_pages += 1

            page_records.append({
                "document_id": document_id,
                "filename": pdf_path.name,
                "source_path": doc_record["source_path"],
                "contract_type": contract_type,
                "page_number": page_num,
                "text": text,
                "is_empty": not bool(stripped),
            })
            doc_record["total_chars"] += len(stripped)

        doc_record["num_empty_pages"] = empty_pages
        doc_record["is_ocr_candidate"] = is_ocr_candidate(None, empty_pages, num_pages)

    except (PdfReadError, Exception) as exc:
        doc_record["parse_status"] = "failed"
        doc_record["parse_error"] = str(exc)
        failures.append({"filename": pdf_path.name, "error": str(exc)})

    records.append(doc_record)

df_documents = pd.DataFrame(records) if 'pd' in dir() else None

Found 510 PDF files under C:\Users\Lenovo\rag-assistant-project\data\CUAD\CUAD_v1\full_contract_pdf


In [4]:
import pandas as pd
import numpy as np

df_documents = pd.DataFrame(records)
df_pages = pd.DataFrame(page_records)

n_found = len(pdf_paths)
n_parsed = int((df_documents["parse_status"] == "success").sum())
n_failed = int((df_documents["parse_status"] == "failed").sum())
total_pages = int(df_documents["num_pages"].sum())
docs_with_pages = df_documents[df_documents["num_pages"] > 0]
avg_pages = float(docs_with_pages["num_pages"].mean()) if len(docs_with_pages) else 0.0
min_pages = int(docs_with_pages["num_pages"].min()) if len(docs_with_pages) else 0
max_pages = int(docs_with_pages["num_pages"].max()) if len(docs_with_pages) else 0
n_docs_with_empty_text = int((df_documents["num_empty_pages"] > 0).sum())
n_ocr_candidates = int(df_documents["is_ocr_candidate"].sum())

inspection_summary = pd.DataFrame([{
    "PDF files found": n_found,
    "Successfully parsed": n_parsed,
    "Failed to parse": n_failed,
    "Total pages": total_pages,
    "Avg pages/doc": round(avg_pages, 2),
    "Min pages": min_pages,
    "Max pages": max_pages,
    "Docs with >=1 empty page": n_docs_with_empty_text,
    "OCR candidates": n_ocr_candidates,
}]).T.rename(columns={0: "value"})

inspection_summary

,value
PDF files found,510.00
Successfully parsed,510.00
Failed to parse,0.00
Total pages,9348.00
Avg pages/doc,18.33
Min pages,1.00
Max pages,154.00
Docs with >=1 empty page,27.00
OCR candidates,1.00


### Inspection results (generated from the cell above — not hard-coded)

Run the cell below to render a Markdown report populated with the actual computed
numbers. This is intentionally a *code* cell producing Markdown output (via
`IPython.display.Markdown`) rather than free-text Markdown, precisely so the reported
numbers cannot drift from what was executed.

In [5]:
from IPython.display import Markdown, display

failure_lines = "\n".join(f"- `{f['filename']}`: {f['error']}" for f in failures) or "- None"

report = f'''
**Dataset inspection summary**

- **Documents found:** {n_found}
- **Successfully parsed:** {n_parsed}
- **Failed to parse:** {n_failed}
- **Total pages:** {total_pages} (avg {avg_pages:.2f}, min {min_pages}, max {max_pages})
- **Formats present:** PDF only (`full_contract_pdf/`); a corresponding
  `full_contract_txt/` directory may also exist as plain-text mirrors but is not used
  as the primary ingestion source per the project spec.
- **Documents with at least one empty/poorly-parsed page:** {n_docs_with_empty_text}
- **Potential OCR candidates** (>=50% of pages extracted no text): {n_ocr_candidates}

**Parse failures:**
{failure_lines}
'''
display(Markdown(report))


**Dataset inspection summary**

- **Documents found:** 510
- **Successfully parsed:** 510
- **Failed to parse:** 0
- **Total pages:** 9348 (avg 18.33, min 1, max 154)
- **Formats present:** PDF only (`full_contract_pdf/`); a corresponding
  `full_contract_txt/` directory may also exist as plain-text mirrors but is not used
  as the primary ingestion source per the project spec.
- **Documents with at least one empty/poorly-parsed page:** 27
- **Potential OCR candidates** (>=50% of pages extracted no text): 1

**Parse failures:**
- None


## 2.2 Chunking Strategy

**Approach:** legal contracts have inconsistent internal structure across CUAD (some
use numbered sections like `1.1`, `ARTICLE III`, others use free-form paragraphs), so
a single structure-aware splitter is not reliable across the whole corpus. We use a
**recursive, size-bounded splitter with overlap** that:

- Prefers to split on paragraph boundaries, then sentence boundaries, then hard
  character limits, only as a last resort — this keeps most clauses intact.
- Carries page and document metadata through so every chunk can always be traced
  back to its exact source page range.
- Uses a **character-based approximation of tokens** (`CHARS_PER_TOKEN_ESTIMATE = 4`),
  which is a reasonable heuristic for English legal prose without adding a tokenizer
  dependency. `CHUNK_SIZE_TOKENS = 1500` / `CHUNK_OVERLAP_TOKENS = 200` are starting
  values chosen because contract clauses (e.g. indemnification, limitation of
  liability) are often several hundred words long — a smaller chunk risks splitting a
  clause in half, while a much larger chunk dilutes retrieval precision. These values
  are configurable in Section 1 and should be tuned against the failure analysis in
  Section 2.6/2.7 rather than assumed optimal.

Each chunk gets a deterministic ID of the form `{document_id}_chunk_{index:04d}`.

In [6]:
import re

def split_into_paragraphs(text: str):
    parts = re.split(r"\n\s*\n", text)
    return [p.strip() for p in parts if p.strip()]


def recursive_chunk_document(doc_pages, chunk_size_chars: int, overlap_chars: int):
    '''doc_pages: list of dicts with keys page_number, text (already extracted).
    Returns a list of chunk dicts with page_start/page_end and chunk_text.
    '''
    # Build a flat list of (page_number, paragraph_text) preserving order
    flat_units = []
    for page in doc_pages:
        for para in split_into_paragraphs(page["text"]):
            flat_units.append((page["page_number"], para))

    if not flat_units:
        return []

    chunks = []
    current_text = ""
    current_pages = set()

    def flush():
        nonlocal current_text, current_pages
        if current_text.strip():
            chunks.append({
                "chunk_text": current_text.strip(),
                "page_start": min(current_pages),
                "page_end": max(current_pages),
            })

    for page_number, para in flat_units:
        candidate = f"{current_text}\n\n{para}".strip() if current_text else para

        if len(candidate) <= chunk_size_chars:
            current_text = candidate
            current_pages.add(page_number)
            continue

        # Paragraph alone is larger than chunk size: hard-split it
        if len(para) > chunk_size_chars:
            flush()
            start = 0
            while start < len(para):
                end = start + chunk_size_chars
                piece = para[start:end]
                chunks.append({
                    "chunk_text": piece.strip(),
                    "page_start": page_number,
                    "page_end": page_number,
                })
                start = end - overlap_chars if end - overlap_chars > start else end
            current_text = ""
            current_pages = set()
            continue

        # Flush current chunk, start new one with overlap tail + this paragraph
        flush()
        tail = current_text[-overlap_chars:] if overlap_chars > 0 else ""
        current_text = f"{tail}\n\n{para}".strip() if tail else para
        current_pages = {page_number}

    flush()
    return [c for c in chunks if c["chunk_text"]]


CHUNK_SIZE_CHARS = CHUNK_SIZE_TOKENS * CHARS_PER_TOKEN_ESTIMATE
CHUNK_OVERLAP_CHARS = CHUNK_OVERLAP_TOKENS * CHARS_PER_TOKEN_ESTIMATE

all_chunks = []
pages_by_doc = df_pages.groupby("document_id")

for _, doc_row in df_documents[df_documents["parse_status"] == "success"].iterrows():
    document_id = doc_row["document_id"]
    if document_id not in pages_by_doc.groups:
        continue
    doc_pages_df = pages_by_doc.get_group(document_id).sort_values("page_number")
    doc_pages = doc_pages_df.to_dict("records")

    doc_chunks = recursive_chunk_document(doc_pages, CHUNK_SIZE_CHARS, CHUNK_OVERLAP_CHARS)
    for chunk_idx, chunk in enumerate(doc_chunks, start=1):
        all_chunks.append({
            "document_id": document_id,
            "filename": doc_row["filename"],
            "source_path": doc_row["source_path"],
            "contract_type": doc_row["contract_type"],
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "chunk_id": f"{document_id}_chunk_{chunk_idx:04d}",
            "chunk_text": chunk["chunk_text"],
        })

df_chunks = pd.DataFrame(all_chunks)
print(f"Total chunks created: {len(df_chunks)}")

Total chunks created: 6650


In [7]:
if len(df_chunks) > 0:
    chunk_lengths = df_chunks["chunk_text"].str.len()
    chunk_stats = pd.DataFrame([{
        "Number of chunks": len(df_chunks),
        "Avg chunk length (chars)": round(chunk_lengths.mean(), 1),
        "Min chunk length (chars)": int(chunk_lengths.min()),
        "Max chunk length (chars)": int(chunk_lengths.max()),
    }]).T.rename(columns={0: "value"})
else:
    chunk_stats = pd.DataFrame({"value": ["No chunks produced — check parsing step above."]})

chunk_stats

,value
Number of chunks,6650.0
Avg chunk length (chars),4713.9
Min chunk length (chars),132.0
Max chunk length (chars),6801.0


In [8]:
# Show a few short representative samples only — never dump full contract text.
SAMPLE_PREVIEW_CHARS = 220
if len(df_chunks) > 0:
    sample = df_chunks.sample(min(3, len(df_chunks)), random_state=42)
    for _, row in sample.iterrows():
        preview = row["chunk_text"][:SAMPLE_PREVIEW_CHARS].replace("\n", " ")
        print(f"[{row['chunk_id']}] ({row['filename']}, pages {row['page_start']}-{row['page_end']})")
        print(f"  {preview}{'...' if len(row['chunk_text']) > SAMPLE_PREVIEW_CHARS else ''}\n")
else:
    print("No chunks available to sample.")

[CUAD_000181_chunk_0040] (VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_EX-10.4_Service Agreement.pdf, pages 43-43)
  Make-Whole Payments due and payable on the Notes, as set forth in Section 10.1(a) of the Indenture. On the Payment Date on which the Optional Redemption is to be exercised, the Issuer shall transfer the entire pool of Re...

[CUAD_000152_chunk_0010] (InmodeLtd_20190729_F-1A_EX-10.9_11743243_EX-10.9_Manufacturing Agreement.pdf, pages 11-11)
  pt written notice of any Manufacturing Claim no later than three (3) business days following receipt of notice by Customer; (ii) Customer will grant Contractor sole control of the defense and settlement of Manufacturing ...

[CUAD_000434_chunk_0029] (KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1-PROMOTION AGREEMENT.PDF, pages 30-30)
  rty matters, safety, FDA, manufacturing or supply issues, or market conditions; and (ii) Depomed shall have no liability under this Agreement for any failure by BLS to timely deliver and supply the 1000mg Fo

## 2.3 Embeddings & Vector Store

We embed chunks with `sentence-transformers` (`all-MiniLM-L6-v2` — a small, fast,
general-purpose model that is a reasonable default absent a domain-tuned legal
embedding model in this environment) and persist them to a local **ChromaDB**
collection at `data/vector_store/`.

**Build vs. Load:** if a populated collection already exists on disk, we load it
instead of recomputing embeddings. This keeps re-runs fast and avoids wasted GPU/CPU
work.

In [9]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

existing_collections = [c.name for c in chroma_client.list_collections()]
collection_exists = COLLECTION_NAME in existing_collections
existing_count = 0
if collection_exists:
    _tmp_collection = chroma_client.get_collection(COLLECTION_NAME)
    existing_count = _tmp_collection.count()

BUILD_MODE = not (collection_exists and existing_count > 0)

if BUILD_MODE:
    print("=" * 60)
    print("BUILDING VECTOR STORE")
    print("=" * 60)
else:
    print("=" * 60)
    print("LOADING EXISTING VECTOR STORE")
    print("=" * 60)
    print(f"Found existing collection '{COLLECTION_NAME}' with {existing_count} items.")

LOADING EXISTING VECTOR STORE
Found existing collection 'cuad_contracts' with 6650 items.


In [10]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Loaded embedding model '{EMBEDDING_MODEL_NAME}' (dim={embedding_dim})")

if BUILD_MODE:
    if collection_exists:
        chroma_client.delete_collection(COLLECTION_NAME)
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"embedding_model": EMBEDDING_MODEL_NAME, "dataset": "CUAD"},
    )

    if len(df_chunks) == 0:
        print("No chunks to embed — skipping population step.")
    else:
        BATCH_SIZE = 64
        n_chunks = len(df_chunks)
        for start in range(0, n_chunks, BATCH_SIZE):
            batch = df_chunks.iloc[start:start + BATCH_SIZE]
            texts = batch["chunk_text"].tolist()
            embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
            collection.add(
                ids=batch["chunk_id"].tolist(),
                embeddings=embeddings,
                documents=texts,
                metadatas=[
                    {
                        "document_id": r["document_id"],
                        "filename": r["filename"],
                        "source_path": r["source_path"],
                        "contract_type": r["contract_type"],
                        "page_start": int(r["page_start"]),
                        "page_end": int(r["page_end"]),
                    }
                    for _, r in batch.iterrows()
                ],
            )
            print(f"  Embedded and stored {min(start + BATCH_SIZE, n_chunks)}/{n_chunks} chunks")

        print(f"\nVector store built: {collection.count()} items in collection '{COLLECTION_NAME}'")
else:
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print(f"Vector store loaded: {collection.count()} items in collection '{COLLECTION_NAME}'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model 'all-MiniLM-L6-v2' (dim=384)
Vector store loaded: 6650 items in collection 'cuad_contracts'


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25388\1931147419.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = embedding_model.get_sentence_embedding_dimension()


## 2.4 Retrieval & Prompting

### Retrieval function

`retrieve_documents(question, top_k)` embeds the query with the same embedding model
used to build the store, queries ChromaDB, and returns chunk text plus full metadata
(document, pages, contract type, source path) and the distance score for each result.

In [11]:
def retrieve_documents(question: str, top_k: int = TOP_K):
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    hits = []
    if results["ids"] and results["ids"][0]:
        for i in range(len(results["ids"][0])):
            hits.append({
                "chunk_id": results["ids"][0][i],
                "chunk_text": results["documents"][0][i],
                "distance": results["distances"][0][i] if results.get("distances") else None,
                **results["metadatas"][0][i],
            })
    return hits

### Legal test questions

At least 10 realistic contract-review questions, spanning distinct clause categories.
These are used for both the retrieval demonstration and evaluation below.

In [12]:
TEST_QUESTIONS = [
    {"id": "q01", "category": "Change of Control",
     "question": "What happens to this agreement if one of the parties undergoes a change of control?"},
    {"id": "q02", "category": "Assignment",
     "question": "Can either party assign this agreement to a third party without consent?"},
    {"id": "q03", "category": "Termination",
     "question": "Under what conditions can either party terminate this agreement for convenience?"},
    {"id": "q04", "category": "Renewal",
     "question": "Does this agreement renew automatically, and how much notice is required to prevent renewal?"},
    {"id": "q05", "category": "Confidentiality",
     "question": "How long must confidential information be kept confidential after disclosure?"},
    {"id": "q06", "category": "Non-compete",
     "question": "Is there a non-compete restriction, and how long does it last after the agreement ends?"},
    {"id": "q07", "category": "Indemnification",
     "question": "What are the indemnification obligations between the parties?"},
    {"id": "q08", "category": "Limitation of Liability",
     "question": "Is there a cap on either party's liability, and what is the amount or formula?"},
    {"id": "q09", "category": "Governing Law",
     "question": "Which state's or country's law governs this agreement?"},
    {"id": "q10", "category": "Intellectual Property",
     "question": "Who owns intellectual property created during performance of this agreement?"},
    {"id": "q11", "category": "Exclusivity",
     "question": "Does this agreement grant exclusive distribution or sales rights, and for what territory?"},
    {"id": "q12", "category": "Insurance",
     "question": "What minimum insurance coverage must be maintained under this agreement?"},
]

pd.DataFrame(TEST_QUESTIONS)

,id,category,question
0,q01,Change of Control,What happens to this agreement if one of the p...
1,q02,Assignment,Can either party assign this agreement to a th...
2,q03,Termination,Under what conditions can either party termina...
3,q04,Renewal,"Does this agreement renew automatically, and h..."
4,q05,Confidentiality,How long must confidential information be kept...
5,q06,Non-compete,"Is there a non-compete restriction, and how lo..."
6,q07,Indemnification,What are the indemnification obligations betwe...
7,q08,Limitation of Liability,"Is there a cap on either party's liability, an..."
8,q09,Governing Law,Which state's or country's law governs this ag...
9,q10,Intellectual Property,Who owns intellectual property created during ...


### Grounded prompt template

The prompt instructs the model to answer strictly from the retrieved context, refuse
to speculate when the answer isn't present, and always cite its sources in the format
`[Source: filename.pdf | Chunk: chunk_id | Pages: start-end]`.

In [13]:
GROUNDED_PROMPT_TEMPLATE = '''You are a contract-review assistant. Answer the question using ONLY the
context excerpts below, which come from real commercial contracts.

Rules:
1. Answer only using the supplied context. Do not use outside knowledge.
2. Do not invent facts, numbers, dates, or parties that are not in the context.
3. If the answer is not present in the context, explicitly say so — do not guess.
4. Clearly distinguish between what the context states as fact and anything uncertain
   or ambiguous in the text.
5. After your answer, cite every source you used, in this exact format:
   [Source: <filename> | Chunk: <chunk_id> | Pages: <page_start>-<page_end>]
6. Never invent a citation. Only cite chunks that were actually provided below.
7. This is not legal advice.

Context:
{context}

Question: {question}

Answer:'''


def build_prompt(question: str, hits: list) -> str:
    context_blocks = []
    for h in hits:
        context_blocks.append(
            f"[Chunk: {h['chunk_id']} | Source: {h['filename']} | "
            f"Pages: {h['page_start']}-{h['page_end']}]\n{h['chunk_text']}"
        )
    context = "\n\n---\n\n".join(context_blocks) if context_blocks else "(no context retrieved)"
    return GROUNDED_PROMPT_TEMPLATE.format(context=context, question=question)

### Ollama generation

We check whether the Ollama service is actually reachable and which models are
installed **before** trying to use it. If Ollama is unavailable, retrieval still runs
in full — generation is simply reported as skipped, never faked.

In [14]:
import ollama as ollama_client

def check_ollama_availability():
    try:
        models_response = ollama_client.list()
        model_names = [m.get("model") or m.get("name") for m in models_response.get("models", [])]
        return True, model_names
    except Exception as exc:
        return False, str(exc)

OLLAMA_AVAILABLE, ollama_info = check_ollama_availability()

if OLLAMA_AVAILABLE:
    available_models = ollama_info
    print(f"Ollama is running. Available models: {available_models}")
    # Prefer a small general-purpose instruct model if present; otherwise take the first.
    preferred = [m for m in available_models if any(k in m.lower() for k in ["llama3", "mistral", "phi3", "qwen"])]
    OLLAMA_MODEL = preferred[0] if preferred else (available_models[0] if available_models else None)
    if OLLAMA_MODEL:
        print(f"Selected OLLAMA_MODEL = '{OLLAMA_MODEL}'")
    else:
        print("Ollama is running but no local models are installed. Generation will be skipped.")
        OLLAMA_AVAILABLE = False
else:
    print(f"Ollama is NOT available: {ollama_info}")
    print("Retrieval-only mode: generation steps below will be clearly marked as skipped.")

Ollama is running. Available models: ['qwen2.5:7b']
Selected OLLAMA_MODEL = 'qwen2.5:7b'


In [15]:
def generate_answer(question: str, hits: list):
    '''Returns (answer_text, was_generated: bool).'''
    prompt = build_prompt(question, hits)
    if not OLLAMA_AVAILABLE or not OLLAMA_MODEL:
        return ("[GENERATION SKIPPED: Ollama is not available in this environment. "
                "Retrieval results above are still valid.]", False)
    try:
        response = ollama_client.generate(model=OLLAMA_MODEL, prompt=prompt)
        return response.get("response", "").strip(), True
    except Exception as exc:
        return (f"[GENERATION FAILED: {exc}]", False)

### Retrieval demonstration

Run every test question through retrieval and (if available) generation. Output is
kept concise — short previews only, never full chunk dumps.

In [16]:
PREVIEW_CHARS = 200
demo_rows = []

for tq in TEST_QUESTIONS:
    hits = retrieve_documents(tq["question"], top_k=TOP_K)
    answer, was_generated = generate_answer(tq["question"], hits)

    print(f"\n{'=' * 80}")
    print(f"[{tq['id']}] ({tq['category']}) {tq['question']}")
    print(f"{'-' * 80}")
    if not hits:
        print("  No results retrieved.")
    for h in hits:
        preview = h["chunk_text"][:PREVIEW_CHARS].replace("\n", " ")
        dist = f"{h['distance']:.4f}" if h["distance"] is not None else "n/a"
        print(f"  - {h['filename']} | pages {h['page_start']}-{h['page_end']} | "
              f"chunk {h['chunk_id']} | distance={dist}")
        print(f"      \"{preview}{'...' if len(h['chunk_text']) > PREVIEW_CHARS else ''}\"")
    print(f"\n  Answer generated: {was_generated}")
    print(f"  Answer preview: {answer[:300]}{'...' if len(answer) > 300 else ''}")

    demo_rows.append({
        "id": tq["id"],
        "category": tq["category"],
        "question": tq["question"],
        "hits": hits,
        "answer": answer,
        "answer_generated": was_generated,
    })


[q01] (Change of Control) What happens to this agreement if one of the parties undergoes a change of control?
--------------------------------------------------------------------------------
  - PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT .PDF | pages 64-66 | chunk CUAD_000419_chunk_0026 | distance=0.7720
      "any part of the Territory, then the rights under this Agreement may not be assigned without the  express consent of the other Party which consent shall not be unreasonably withheld. “Change of Control..."
  - WESTERN COPPER - NON-COMPETITION AGREEMENT.PDF | pages 7-8 | chunk CUAD_000164_chunk_0005 | distance=0.8301
      "the next succeeding Business Day if delivered or transmitted  subsequent to such time; (c) Either  party hereto may change its address for service from time to time by notice  given to the other party..."
  - SENMIAOTECHNOLOGYLTD_02_19_2019-EX-10.5-Collaboration Agreement.PDF | pages 20-20 | chunk CUAD_000209_c

## 2.5 Vision Component *[Extended]*

**Status: not implemented in this notebook.**

This project's Core Track is text-based RAG over machine-readable CUAD PDFs. A
vision component is explicitly marked `[Extended]` in the project spec and is
**not required** for the Core Track to be considered complete — and, per the
inspection in Section 2.1, this run found `{n_ocr_candidates}` potential OCR/image
candidates that would need one.

**Why not implemented here:**
- No scanned/image-only contract subset or labeled image dataset is available
  locally to build or validate a vision model against.
- No local vision-capable model (e.g. a layout-aware OCR or document classification
  model) is confirmed installed in this environment.
- Fabricating a vision pipeline or made-up accuracy numbers without real data/model
  would violate the "never fabricate results" engineering rule for this project.

**What it would process, if implemented:** scanned contract pages that failed text
extraction (see `is_ocr_candidate` in Section 2.1) — e.g. faxed or image-only PDFs —
using OCR (e.g. Tesseract, or a vision-language model) to recover page text, plus
optionally a layout/classification model to detect structural elements (signature
blocks, tables, letterhead) that a plain text extractor misses.

**How it would plug into this RAG pipeline:** OCR output would be inserted at the
same point where `page.extract_text()` is called in Section 2.1 — i.e. as a fallback
when native text extraction returns an empty/low-confidence result for a page. The
recovered text would then flow through the exact same chunking → embedding → ChromaDB
pipeline already built above, with an added `extraction_method: "ocr"` metadata field
so retrieval results and citations could indicate when an answer relies on
OCR-recovered (lower-confidence) text.

The Core Track above remains fully functional with this component absent.

## 2.6 Evaluation

For each test question we evaluate four dimensions:

- **Retrieval quality** — is at least one retrieved chunk actually relevant to the
  question, checked against CUAD ground truth where available in
  `master_clauses.csv` / `CUAD_v1.json`, otherwise assessed manually against the
  chunk text itself.
- **Answer quality / Grounding** — does the generated answer's content actually
  appear supported by the retrieved context (checked automatically only when
  generation ran; otherwise marked as not applicable).
- **Citation** — does the generated answer include a citation in the required
  format, referencing a chunk that was actually retrieved (never a fabricated one)?

Where CUAD ground truth cannot resolve a question automatically, this is explicitly
marked `manual` rather than silently treated as correct.

In [17]:
# --- Load CUAD ground-truth annotations (used only for evaluation, never ingested
#     into the RAG knowledge base itself) ---------------------------------------
master_clauses_df = None
cuad_json_data = None

if MASTER_CLAUSES_CSV.exists():
    master_clauses_df = pd.read_csv(MASTER_CLAUSES_CSV)
    print(f"Loaded master_clauses.csv with {len(master_clauses_df)} rows, "
          f"{len(master_clauses_df.columns)} columns")
else:
    print(f"master_clauses.csv not found at {MASTER_CLAUSES_CSV} — automatic "
          f"ground-truth matching will be skipped; all questions fall back to manual review.")

if CUAD_JSON_PATH.exists():
    with open(CUAD_JSON_PATH) as f:
        cuad_json_data = json.load(f)
    print(f"Loaded CUAD_v1.json with {len(cuad_json_data.get('data', []))} contract entries")
else:
    print(f"CUAD_v1.json not found at {CUAD_JSON_PATH} — skipping.")

Loaded master_clauses.csv with 510 rows, 83 columns
Loaded CUAD_v1.json with 510 contract entries


In [18]:
# Map each test question's category to the master_clauses.csv column(s) that most
# plausibly contain ground truth for it. Matching is done by fuzzy substring match
# on column names so this works even if exact CUAD column names differ slightly.
CATEGORY_TO_COLUMN_HINTS = {
    "Change of Control": ["change of control", "change-of-control"],
    "Assignment": ["assignment", "anti-assignment"],
    "Termination": ["termination"],
    "Renewal": ["renewal", "auto-renewal", "auto renewal"],
    "Confidentiality": ["confidential"],
    "Non-compete": ["non-compete", "noncompete"],
    "Indemnification": ["indemnif"],
    "Limitation of Liability": ["liability", "cap on liability", "uncapped liability"],
    "Governing Law": ["governing law"],
    "Intellectual Property": ["ip ownership", "intellectual property", "joint ip"],
    "Exclusivity": ["exclusiv"],
    "Insurance": ["insurance"],
}


def find_ground_truth_column(category: str, columns) -> str | None:
    hints = CATEGORY_TO_COLUMN_HINTS.get(category, [])
    lower_cols = {c.lower(): c for c in columns}
    for hint in hints:
        for lower_col, orig_col in lower_cols.items():
            if hint in lower_col:
                return orig_col
    return None


def ground_truth_for_hit(hit: dict, category: str):
    '''Look up whether master_clauses.csv has a non-empty ground-truth clause
    entry for this document + category. Returns (has_column, gt_text_or_none).
    '''
    if master_clauses_df is None:
        return (False, None)
    col = find_ground_truth_column(category, master_clauses_df.columns)
    if col is None:
        return (False, None)
    row = master_clauses_df[master_clauses_df["Filename"] == hit["filename"]]
    if row.empty:
        return (True, None)
    value = row.iloc[0][col]
    if pd.isna(value) or str(value).strip() in ("", "[]"):
        return (True, None)
    return (True, str(value))

In [19]:
def keyword_overlap_relevance(chunk_text: str, ground_truth_text: str) -> bool:
    '''Very lightweight automatic relevance check: do the retrieved chunk and the
    CUAD ground-truth clause text share a meaningful run of words? This is not a
    substitute for human review — it is one automatable signal among several.
    '''
    gt_words = [w.lower() for w in re.findall(r"[a-zA-Z]{5,}", ground_truth_text)]
    chunk_lower = chunk_text.lower()
    if not gt_words:
        return False
    matches = sum(1 for w in set(gt_words) if w in chunk_lower)
    return (matches / max(len(set(gt_words)), 1)) >= 0.25


eval_rows = []

for row in demo_rows:
    hits = row["hits"]
    top_hit = hits[0] if hits else None

    # --- Retrieval quality ---
    has_gt_column = False
    gt_text = None
    retrieval_relevant = None
    retrieval_eval_method = "manual"

    if top_hit is not None:
        has_gt_column, gt_text = ground_truth_for_hit(top_hit, row["category"])
        if has_gt_column and gt_text:
            retrieval_relevant = any(
                keyword_overlap_relevance(h["chunk_text"], gt_text) for h in hits
            )
            retrieval_eval_method = "automatic (CUAD ground truth keyword overlap)"
        else:
            retrieval_relevant = None  # requires manual review
            retrieval_eval_method = "manual (no matching ground-truth column/value found)"
    else:
        retrieval_relevant = False
        retrieval_eval_method = "automatic (no chunks retrieved)"

    # --- Citation check ---
    citation_pattern = re.compile(r"\[Source:.*?\|\s*Chunk:\s*(\S+?)\s*\|\s*Pages:", re.IGNORECASE)
    cited_chunk_ids = citation_pattern.findall(row["answer"])
    retrieved_chunk_ids = {h["chunk_id"] for h in hits}
    citation_present = len(cited_chunk_ids) > 0
    citation_valid = citation_present and all(cid in retrieved_chunk_ids for cid in cited_chunk_ids)

    # --- Grounding / answer quality ---
    if not row["answer_generated"]:
        answer_grounded = None
        grounding_method = "not applicable (generation skipped — Ollama unavailable)"
    else:
        combined_context = " ".join(h["chunk_text"].lower() for h in hits)
        answer_words = [w.lower() for w in re.findall(r"[a-zA-Z]{5,}", row["answer"])]
        overlap = sum(1 for w in set(answer_words) if w in combined_context)
        answer_grounded = (overlap / max(len(set(answer_words)), 1)) >= 0.3 if answer_words else False
        grounding_method = "automatic (lexical overlap with retrieved context)"

    correct = retrieval_relevant if retrieval_relevant is not None else None

    eval_rows.append({
        "question_id": row["id"],
        "category": row["category"],
        "question": row["question"],
        "expected_answer_source": "CUAD master_clauses.csv" if (has_gt_column and gt_text) else "N/A (manual review required)",
        "retrieved_source": top_hit["filename"] if top_hit else None,
        "retrieved_chunk": top_hit["chunk_id"] if top_hit else None,
        "retrieved_context_preview": (top_hit["chunk_text"][:150] + "...") if top_hit else None,
        "generated_answer_preview": row["answer"][:150] + ("..." if len(row["answer"]) > 150 else ""),
        "answer_generated": row["answer_generated"],
        "retrieval_relevant": retrieval_relevant,
        "retrieval_eval_method": retrieval_eval_method,
        "answer_grounded": answer_grounded,
        "grounding_eval_method": grounding_method,
        "citation_present": citation_present,
        "citation_valid": citation_valid,
        "correct": correct,
    })

df_eval = pd.DataFrame(eval_rows)
df_eval

,question_id,category,question,expected_answer_source,retrieved_source,retrieved_chunk,retrieved_context_preview,generated_answer_preview,answer_generated,retrieval_relevant,retrieval_eval_method,answer_grounded,grounding_eval_method,citation_present,citation_valid,correct
0,q01,Change of Control,What happens to this agreement if one of the p...,N/A (manual review required),"PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC L...",CUAD_000419_chunk_0026,"any part of the Territory, then the rights und...","If either party undergoes a change of control,...",True,None,manual (no matching ground-truth column/value ...,True,automatic (lexical overlap with retrieved cont...,False,False,None
1,q02,Assignment,Can either party assign this agreement to a th...,CUAD master_clauses.csv,CHANGEPOINTCORP_03_08_2000-EX-10.6-LICENSE AND...,CUAD_000356_chunk_0018,"reement without such consent, except in the ca...","Based on the agreement text, neither party has...",True,True,automatic (CUAD ground truth keyword overlap),True,automatic (lexical overlap with retrieved cont...,False,False,True
2,q03,Termination,Under what conditions can either party termina...,N/A (manual review required),PhoenixNewMediaLtd_20110421_F-1_EX-10.17_69583...,CUAD_000131_chunk_0004,-breaching Party may terminate this Agreement;...,"Based on the provided document, there are spec...",True,None,manual (no matching ground-truth column/value ...,True,automatic (lexical overlap with retrieved cont...,False,False,None
3,q04,Renewal,"Does this agreement renew automatically, and h...",CUAD master_clauses.csv,NATIONALPROCESSINGINC_07_18_1996-EX-10.4-SPONS...,CUAD_000463_chunk_0006,n notice to the other party effective at the l...,"Yes, the agreement automatically renews for on...",True,True,automatic (CUAD ground truth keyword overlap),True,automatic (lexical overlap with retrieved cont...,False,False,True
4,q05,Confidentiality,How long must confidential information be kept...,N/A (manual review required),EtonPharmaceuticalsInc_20191114_10-Q_EX-10.1_1...,CUAD_000040_chunk_0010,be accorded the Confidential Information. Conf...,"According to the agreement, confidential infor...",True,None,manual (no matching ground-truth column/value ...,True,automatic (lexical overlap with retrieved cont...,False,False,None
5,q06,Non-compete,"Is there a non-compete restriction, and how lo...",CUAD master_clauses.csv,"GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...",CUAD_000347_chunk_0050,s defined below); and (e) restrictions on your...,There is no explicit mention of a non-compete ...,True,True,automatic (CUAD ground truth keyword overlap),True,automatic (lexical overlap with retrieved cont...,False,False,True
6,q07,Indemnification,What are the indemnification obligations betwe...,N/A (manual review required),CYBERIANOUTPOSTINC_07_09_1998-EX-10.13-PROMOTI...,CUAD_000433_chunk_0005,", the Indemnified Party shall permit the other...",The indemnification obligations between the pa...,True,None,manual (no matching ground-truth column/value ...,True,automatic (lexical overlap with retrieved cont...,False,False,None
7,q08,Limitation of Liability,"Is there a cap on either party's liability, an...",CUAD master_clauses.csv,ReynoldsConsumerProductsInc_20191115_S-1_EX-10...,CUAD_000193_chunk_0006,from the other Party as provided in this Agree...,"Yes, there is a cap on either party's liabilit...",True,True,automatic (CUAD ground truth keyword overlap),True,automatic (lexical overlap with retrieved cont...,False,False,True
8,q09,Governing Law,Which state's or country's law governs this ag...,CUAD master_clauses.csv,EcoScienceSolutionsInc_20180406_8-K_EX-10.1_11...,CUAD_000184_chunk_0003,arrants and represents that it shall comply wi...,"The agreement is governed by, and construed in...",True,True,automatic (CUAD ground truth keyword overlap),True,automatic (lexical overlap with retrieved cont...,False,False,True
9,q10,Intellectual Property,Who owns intellectual property created during ...,CUAD master_clauses.cs

### Retrieval metrics (Hit@K)

We compute `Hit@1`, `Hit@3`, and `Hit@5` **only** using questions where CUAD ground
truth was actually found for the top result's document (`retrieval_eval_method`
starts with `automatic`), since Hit@K is meaningless without a real ground-truth
signal. Questions requiring manual review are excluded from these metrics and
reported separately — we do not fabricate a metric for them.

- **Hit@K** = fraction of evaluable questions where a ground-truth-matching chunk
  appears within the top-K retrieved results.
- **Recall@K** here is equivalent to Hit@K in this single-relevant-chunk-type setup
  (we treat "does any retrieved chunk in the top K match the ground truth clause" as
  the success criterion, since CUAD does not provide a graded multi-relevant-passage
  list for these free-text questions).

In [20]:
def hit_at_k(question_row_hits, ground_truth_text, k):
    return any(keyword_overlap_relevance(h["chunk_text"], ground_truth_text) for h in question_row_hits[:k])


automatic_rows = []
for row in demo_rows:
    has_gt_column, gt_text = (False, None)
    if row["hits"]:
        has_gt_column, gt_text = ground_truth_for_hit(row["hits"][0], row["category"])
    if has_gt_column and gt_text:
        automatic_rows.append((row, gt_text))

n_evaluable = len(automatic_rows)

if n_evaluable > 0:
    hit_1 = sum(hit_at_k(r["hits"], gt, 1) for r, gt in automatic_rows) / n_evaluable
    hit_3 = sum(hit_at_k(r["hits"], gt, 3) for r, gt in automatic_rows) / n_evaluable
    hit_5 = sum(hit_at_k(r["hits"], gt, 5) for r, gt in automatic_rows) / n_evaluable

    metrics_df = pd.DataFrame([{
        "Evaluable questions (automatic GT match)": n_evaluable,
        "Questions requiring manual review": len(demo_rows) - n_evaluable,
        "Hit@1": round(hit_1, 3),
        "Hit@3": round(hit_3, 3),
        "Hit@5": round(hit_5, 3),
    }]).T.rename(columns={0: "value"})
else:
    metrics_df = pd.DataFrame({
        "value": ["No questions had an automatically resolvable CUAD ground-truth match "
                  "in this dataset/run — all questions require manual retrieval review. "
                  "This is reported honestly rather than substituting a fabricated metric."]
    })

metrics_df

,value
Evaluable questions (automatic GT match),7.0
Questions requiring manual review,5.0
Hit@1,1.0
Hit@3,1.0
Hit@5,1.0


### Failure analysis

Real failures observed in the evaluation above (only populated from what actually
occurred when this notebook was run — never invented).

In [21]:
failure_analysis = []

for row in eval_rows:
    if row["retrieval_relevant"] is False:
        failure_analysis.append({
            "question_id": row["question_id"],
            "failure_type": "Relevant clause not retrieved / no ground-truth match in top-K",
            "detail": f"Top retrieved source: {row['retrieved_source']}",
            "suggested_mitigation": "Try hybrid search (BM25 + dense), query expansion, or a larger top-k.",
        })
    # Citation checks only make sense when generation actually ran. If Ollama was
    # unavailable and generation was correctly skipped, an absent citation is
    # expected behavior, not a pipeline failure — so we gate on answer_generated.
    if row["answer_generated"]:
        if row["citation_present"] and not row["citation_valid"]:
            failure_analysis.append({
                "question_id": row["question_id"],
                "failure_type": "Citation did not correspond to an actually retrieved chunk",
                "detail": "Generated citation referenced a chunk ID not present in retrieval results.",
                "suggested_mitigation": "Constrain generation to copy chunk IDs verbatim from context; "
                                         "post-process to strip/flag invalid citations.",
            })
        elif row["citation_present"] is False and row["retrieved_chunk"] is not None:
            failure_analysis.append({
                "question_id": row["question_id"],
                "failure_type": "No citation present despite retrieved context being available",
                "detail": "Generated answer lacked the required citation format.",
                "suggested_mitigation": "Strengthen prompt instructions; consider few-shot citation examples "
                                         "or a citation-formatting post-processing step.",
            })
    if row["answer_grounded"] is False:
        failure_analysis.append({
            "question_id": row["question_id"],
            "failure_type": "Generated answer had low lexical overlap with retrieved context (possible hallucination)",
            "detail": row["generated_answer_preview"],
            "suggested_mitigation": "Lower generation temperature, add stricter grounding instructions, "
                                     "or add a post-hoc groundedness classifier.",
        })

df_failures = pd.DataFrame(failure_analysis)
if not OLLAMA_AVAILABLE:
    print("Note: Ollama was unavailable, so generation-dependent failure checks "
          "(citation format, grounding) were skipped for all questions rather than "
          "scored — only retrieval-side failures are assessed below.")
if len(df_failures) == 0:
    print("No failures matching the tracked categories occurred in this run.")
else:
    print(f"{len(df_failures)} failure instance(s) recorded:")
df_failures

12 failure instance(s) recorded:


,question_id,failure_type,detail,suggested_mitigation
0,q01,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
1,q02,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
2,q03,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
3,q04,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
4,q05,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
5,q06,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
6,q07,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
7,q08,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
8,q09,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...
9,q10,No citation present despite retrieved context ...,Generated answer lacked the required citation ...,Strengthen prompt instructions; consider few-s...


## 2.7 Export

We persist a `config.json` alongside the ChromaDB store so Phase 3 (FastAPI backend)
can load the existing vector store **without recomputing embeddings**. All values are
taken from this run's actual execution state.

In [22]:
export_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dim": embedding_dim,
    "chunk_size_tokens": CHUNK_SIZE_TOKENS,
    "chunk_overlap_tokens": CHUNK_OVERLAP_TOKENS,
    "chars_per_token_estimate": CHARS_PER_TOKEN_ESTIMATE,
    "vector_store": "chromadb",
    "vector_store_path": str(VECTOR_STORE_DIR.relative_to(PROJECT_ROOT)),
    "collection_name": COLLECTION_NAME,
    "dataset": "CUAD",
    "document_count": int((df_documents["parse_status"] == "success").sum()),
    "document_count_total_found": int(len(df_documents)),
    "chunk_count": int(len(df_chunks)),
    "top_k_default": TOP_K,
    "ollama_model_used": OLLAMA_MODEL if OLLAMA_AVAILABLE else None,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(VECTOR_STORE_CONFIG_PATH, "w") as f:
    json.dump(export_config, f, indent=2)

print(f"Wrote config to {VECTOR_STORE_CONFIG_PATH}")
print(json.dumps(export_config, indent=2))

Wrote config to C:\Users\Lenovo\rag-assistant-project\data\vector_store\config.json
{
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "chunk_size_tokens": 1500,
  "chunk_overlap_tokens": 200,
  "chars_per_token_estimate": 4,
  "vector_store": "chromadb",
  "vector_store_path": "data\\vector_store",
  "collection_name": "cuad_contracts",
  "dataset": "CUAD",
  "document_count": 510,
  "document_count_total_found": 510,
  "chunk_count": 6650,
  "top_k_default": 5,
  "ollama_model_used": "qwen2.5:7b",
  "created_at": "2026-09-09T21:50:01.936368+00:00"
}


In [23]:
# Verify persistence: re-open a fresh Chroma client pointed at the same directory
# and confirm the collection + item count are actually on disk.
verify_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
verify_collection = verify_client.get_collection(COLLECTION_NAME)
print(f"Verification: collection '{COLLECTION_NAME}' persisted with "
      f"{verify_collection.count()} items at {VECTOR_STORE_DIR}")
assert verify_collection.count() == collection.count(), "Persisted count mismatch!"
print("Persistence verified OK.")

Verification: collection 'cuad_contracts' persisted with 6650 items at C:\Users\Lenovo\rag-assistant-project\data\vector_store
Persistence verified OK.


## Phase 2 Summary

Run the cell below to render a summary populated entirely from this notebook's own
execution state (no manually written numbers).

In [24]:
n_manual_review = sum(1 for r in eval_rows if r["retrieval_eval_method"].startswith("manual"))

summary = f'''
1. **Dataset inspection:** {n_found} PDF files found under `full_contract_pdf/`;
   {n_parsed} parsed successfully, {n_failed} failed; {total_pages} total pages
   ({n_docs_with_empty_text} documents had at least one empty page;
   {n_ocr_candidates} flagged as OCR candidates).
2. **PDF extraction:** `pypdf`, per-page, with deterministic `CUAD_XXXXXX` document IDs.
3. **Chunking strategy:** recursive paragraph-aware splitter,
   ~{CHUNK_SIZE_TOKENS} token chunks / ~{CHUNK_OVERLAP_TOKENS} token overlap,
   producing {len(df_chunks)} chunks with page-range metadata preserved.
4. **Embedding model:** `{EMBEDDING_MODEL_NAME}` (dim={embedding_dim}).
5. **Vector database:** ChromaDB, collection `{COLLECTION_NAME}`, persisted at
   `{VECTOR_STORE_DIR.relative_to(PROJECT_ROOT)}`.
6. **Retrieval approach:** dense similarity search via `retrieve_documents()`,
   default top_k={TOP_K}.
7. **LLM generation:** Ollama {'was available; model used: ' + str(OLLAMA_MODEL) if OLLAMA_AVAILABLE else 'was NOT available in this environment — generation was skipped and clearly marked; retrieval remained fully functional.'}
8. **Evaluation results:** {len(eval_rows)} test questions evaluated across
   {len(TEST_QUESTIONS)} clause categories; {len(eval_rows) - n_manual_review} had an
   automatic CUAD ground-truth match, {n_manual_review} require manual review.
   {"Hit@1/@3/@5 = " + f"{hit_1:.2f}/{hit_3:.2f}/{hit_5:.2f}" if n_evaluable > 0 else "No automatically-evaluable Hit@K could be computed on this run's data."}
9. **Failure cases:** {len(df_failures)} recorded (see failure analysis table above)
   with concrete mitigations proposed per case.
10. **Exported artifacts:** `{VECTOR_STORE_DIR.relative_to(PROJECT_ROOT)}` (ChromaDB store)
    and `{VECTOR_STORE_CONFIG_PATH.relative_to(PROJECT_ROOT)}`.
11. **Limitations:** automatic grounding/citation checks use lexical-overlap
    heuristics, not a certified NLI/entailment model; Hit@K only covers questions
    with a resolvable CUAD ground-truth column; the embedding model is a
    general-purpose default, not a legal-domain fine-tuned model; Vision `[Extended]`
    is not implemented.
12. **Recommended next steps for Phase 3:** load `config.json` + the persisted
    ChromaDB store directly (no re-embedding) in the FastAPI backend; consider
    hybrid (BM25 + dense) retrieval and/or reranking if Hit@K on the full dataset is
    low; add a real NLI-based groundedness check before shipping generated answers;
    revisit chunk size using the full-corpus failure analysis once run at scale.
'''
display(Markdown(summary))


1. **Dataset inspection:** 510 PDF files found under `full_contract_pdf/`;
   510 parsed successfully, 0 failed; 9348 total pages
   (27 documents had at least one empty page;
   1 flagged as OCR candidates).
2. **PDF extraction:** `pypdf`, per-page, with deterministic `CUAD_XXXXXX` document IDs.
3. **Chunking strategy:** recursive paragraph-aware splitter,
   ~1500 token chunks / ~200 token overlap,
   producing 6650 chunks with page-range metadata preserved.
4. **Embedding model:** `all-MiniLM-L6-v2` (dim=384).
5. **Vector database:** ChromaDB, collection `cuad_contracts`, persisted at
   `data\vector_store`.
6. **Retrieval approach:** dense similarity search via `retrieve_documents()`,
   default top_k=5.
7. **LLM generation:** Ollama was available; model used: qwen2.5:7b
8. **Evaluation results:** 12 test questions evaluated across
   12 clause categories; 7 had an
   automatic CUAD ground-truth match, 5 require manual review.
   Hit@1/@3/@5 = 1.00/1.00/1.00
9. **Failure cases:** 12 recorded (see failure analysis table above)
   with concrete mitigations proposed per case.
10. **Exported artifacts:** `data\vector_store` (ChromaDB store)
    and `data\vector_store\config.json`.
11. **Limitations:** automatic grounding/citation checks use lexical-overlap
    heuristics, not a certified NLI/entailment model; Hit@K only covers questions
    with a resolvable CUAD ground-truth column; the embedding model is a
    general-purpose default, not a legal-domain fine-tuned model; Vision `[Extended]`
    is not implemented.
12. **Recommended next steps for Phase 3:** load `config.json` + the persisted
    ChromaDB store directly (no re-embedding) in the FastAPI backend; consider
    hybrid (BM25 + dense) retrieval and/or reranking if Hit@K on the full dataset is
    low; add a real NLI-based groundedness check before shipping generated answers;
    revisit chunk size using the full-corpus failure analysis once run at scale.
